In [1]:
import pandas as pd
import unicodedata
import re
import numpy as np
from transformers import EarlyStoppingCallback
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [2]:
train_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/new-dataset/merged_train.csv")
test_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/new-dataset/merged_test.csv")
valid_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/new-dataset/merged_valid.csv")

In [3]:
jap_valid=pd.read_csv("/kaggle/input/datasets/poushalisanyal/jap-validation/japenese_valid.csv")
jap_test=pd.read_csv("/kaggle/input/datasets/poushalisanyal/jap-test/japnese_test.csv")

In [4]:
chinese_test=pd.read_csv("/kaggle/input/datasets/poushalisanyal/chinese/chinese_test.csv")
chinese_valid=pd.read_csv("/kaggle/input/datasets/poushalisanyal/chinese/chinese_valid.csv")

In [5]:
train_df.head()

,text,label,source,lang
0,RT @user: @user @user وصلنا لاقتصاد اسوء من ...,negative,sem_eval_2017,ar
1,كاني ويست، دريك، نيكي، بيونسيه، قاقا http,neutral,sem_eval_2017,ar
2,@user على فكره شركة محترمه حداعطوني كيبل كهديه...,positive,sem_eval_2017,ar
3,RT @user: المتعه افضل من الزواج لهتك اعراض عام...,negative,sem_eval_2017,ar
4,القوات البرية السعودية والقوات الفرنسية الخاصة...,neutral,sem_eval_2017,ar


In [6]:
jap_valid.shape

(3000, 3)

In [7]:
jap_test.shape

(3000, 3)

In [8]:
train_df.shape

(30399, 4)

In [9]:
test_df.shape

(8465, 4)

In [10]:
valid_df.shape

(4857, 4)

In [11]:
chinese_test.shape

(3000, 3)

In [12]:
chinese_valid.shape

(3000, 3)

In [13]:
labels = sorted(train_df["label"].unique())

label2id = {v: i for i, v in enumerate(labels)}
id2label = {i: v for v, i in label2id.items()}

train_df["label"] = train_df["label"].map(label2id)
valid_df["label"] = valid_df["label"].map(label2id)
test_df["label"] = test_df["label"].map(label2id)
jap_valid["label"] = jap_valid["label"].map(label2id)
jap_test["label"] = jap_test["label"].map(label2id)
chinese_test["label"] = chinese_test["label"].map(label2id)
chinese_valid["label"] = chinese_valid["label"].map(label2id)

In [14]:
def normalize(text):
    return unicodedata.normalize("NFKC", str(text))

def remove_noise(text):
    text = re.sub(r"http\S+|www\S+", "", text)   # links
    text = re.sub(r"@\w+", "", text)             # mentions
    text = re.sub(r"\s+", " ", text).strip()     # spaces
    return text

def fix_punct(text):
    return re.sub(r"([!?.,])\1+", r"\1", text)

def fix_repeating(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def remove_emojis(text):
    emoji_pattern = re.compile("["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE)
    return emoji_pattern.sub("", text)

In [15]:
train_df["text"] = train_df["text"].apply(remove_noise)
train_df["text"] = train_df["text"].apply(normalize)
train_df["text"] = train_df["text"].apply(fix_punct)
train_df["text"] = train_df["text"].apply(fix_repeating)
train_df["text"] = train_df["text"].apply(remove_emojis)

In [16]:
test_df["text"] = test_df["text"].apply(remove_noise)
test_df["text"] = test_df["text"].apply(normalize)
test_df["text"] = test_df["text"].apply(fix_punct)
test_df["text"] = test_df["text"].apply(fix_repeating)
test_df["text"] = test_df["text"].apply(remove_emojis)

In [17]:
valid_df["text"] = valid_df["text"].apply(remove_noise)
valid_df["text"] = valid_df["text"].apply(normalize)
valid_df["text"] = valid_df["text"].apply(fix_punct)
valid_df["text"] = valid_df["text"].apply(fix_repeating)
valid_df["text"] = valid_df["text"].apply(remove_emojis)

In [18]:
jap_valid["text"] = jap_valid["text"].apply(remove_noise)
jap_valid["text"] = jap_valid["text"].apply(normalize)
jap_valid["text"] = jap_valid["text"].apply(fix_punct)
jap_valid["text"] = jap_valid["text"].apply(fix_repeating)
jap_valid["text"] = jap_valid["text"].apply(remove_emojis)

In [19]:
jap_test["text"] = jap_test["text"].apply(remove_noise)
jap_test["text"] = jap_test["text"].apply(normalize)
jap_test["text"] = jap_test["text"].apply(fix_punct)
jap_test["text"] = jap_test["text"].apply(fix_repeating)
jap_test["text"] = jap_test["text"].apply(remove_emojis)

In [20]:
chinese_valid["text"] = chinese_valid["text"].apply(remove_noise)
chinese_valid["text"] = chinese_valid["text"].apply(normalize)
chinese_valid["text"] = chinese_valid["text"].apply(fix_punct)
chinese_valid["text"] = chinese_valid["text"].apply(fix_repeating)
chinese_valid["text"] = chinese_valid["text"].apply(remove_emojis)

In [21]:
chinese_test["text"] = chinese_test["text"].apply(remove_noise)
chinese_test["text"] = chinese_test["text"].apply(normalize)
chinese_test["text"] = chinese_test["text"].apply(fix_punct)
chinese_test["text"] = chinese_test["text"].apply(fix_repeating)
chinese_test["text"] = chinese_test["text"].apply(remove_emojis)

In [22]:
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [23]:
def tokenize_fn(ex):
    return tokenizer(
        ex["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [25]:
from transformers import AutoModelForSequenceClassification

num_labels = 3
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=num_labels
)

# increase dropout
model.config.hidden_dropout_prob = 0.3
model.config.attention_probs_dropout_prob = 0.3

# freeze embedding layer 
for param in model.bert.embeddings.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))
jap_ds = Dataset.from_pandas(jap_valid.reset_index(drop=True))
jap_test = Dataset.from_pandas(jap_test.reset_index(drop=True))
chinese_test = Dataset.from_pandas(chinese_test.reset_index(drop=True))
chinese_valid = Dataset.from_pandas(chinese_valid.reset_index(drop=True))

train_tokd = train_ds.map(tokenize_fn, batched=True)
test_tokd = test_ds.map(tokenize_fn, batched=True)
val_tokd = val_ds.map(tokenize_fn, batched=True)
jap_tokd = jap_ds.map(tokenize_fn, batched=True)
jap_test_tokd = jap_test.map(tokenize_fn, batched=True)
jap_test_tokd = jap_test.map(tokenize_fn, batched=True)
jap_test_tokd = jap_test.map(tokenize_fn, batched=True)
chinese_test_tokd = chinese_test.map(tokenize_fn, batched=True)
chinese_valid_tokd = chinese_valid.map(tokenize_fn, batched=True)

Map:   0%|          | 0/30399 [00:00<?, ? examples/s]

Map:   0%|          | 0/8465 [00:00<?, ? examples/s]

Map:   0%|          | 0/4857 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [27]:
!pip install evaluate
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [28]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

   
    preds = np.argmax(logits, axis=-1)

    
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

In [29]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [30]:
training_args = TrainingArguments(
    output_dir="sentiment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    weight_decay=0.01,
    label_smoothing_factor=0.0,
    max_grad_norm=1.0,
    fp16=True,
    save_total_limit=2
)

In [31]:
trainer = Trainer(model=model,
    args=training_args,
    train_dataset=train_tokd,
    eval_dataset=val_tokd,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [32]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.484904,0.660078,0.660945,0.660078,0.658743
2,3.215892,1.401895,0.696109,0.702409,0.696109,0.692852
3,2.356389,1.452803,0.697550,0.695871,0.697550,0.693588
4,1.861566,1.594939,0.700021,0.705282,0.700021,0.702059
5,1.486467,1.695841,0.694050,0.695657,0.694050,0.694586
6,1.138180,1.844478,0.698785,0.699689,0.698785,0.699213


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2850, training_loss=1.8857401637027138, metrics={'train_runtime': 2964.859, 'train_samples_per_second': 61.519, 'train_steps_per_second': 0.961, 'total_flos': 1.1997577178270208e+16, 'train_loss': 1.8857401637027138, 'epoch': 6.0})

In [33]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.595678448677063,
 'eval_accuracy': 0.7000205888408483,
 'eval_precision': 0.7052821560351044,
 'eval_recall': 0.7000205888408483,
 'eval_f1': 0.7020594242913772,
 'eval_runtime': 29.2678,
 'eval_samples_per_second': 165.95,
 'eval_steps_per_second': 5.193,
 'epoch': 6.0}

In [34]:
test_results = trainer.evaluate(test_tokd)
print(test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.8818238973617554, 'eval_accuracy': 0.6378027170702895, 'eval_precision': 0.6432348994720489, 'eval_recall': 0.6378027170702895, 'eval_f1': 0.639216772955777, 'eval_runtime': 51.0695, 'eval_samples_per_second': 165.755, 'eval_steps_per_second': 5.189, 'epoch': 6.0}


In [35]:
valid_results = trainer.evaluate(val_tokd)
print(valid_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.595678448677063, 'eval_accuracy': 0.7000205888408483, 'eval_precision': 0.7052821560351044, 'eval_recall': 0.7000205888408483, 'eval_f1': 0.7020594242913772, 'eval_runtime': 29.1256, 'eval_samples_per_second': 166.76, 'eval_steps_per_second': 5.219, 'epoch': 6.0}


In [36]:
jap_results = trainer.evaluate(jap_tokd)
print(jap_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 3.0520150661468506, 'eval_accuracy': 0.5026666666666667, 'eval_precision': 0.4972022502318796, 'eval_recall': 0.5026666666666667, 'eval_f1': 0.4439527859740279, 'eval_runtime': 18.1238, 'eval_samples_per_second': 165.528, 'eval_steps_per_second': 5.187, 'epoch': 6.0}


In [37]:
japen_test = trainer.evaluate(jap_test_tokd)
print(japen_test)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 3.1037204265594482, 'eval_accuracy': 0.48, 'eval_precision': 0.4709631500088455, 'eval_recall': 0.48, 'eval_f1': 0.42335609795014906, 'eval_runtime': 18.2706, 'eval_samples_per_second': 164.198, 'eval_steps_per_second': 5.145, 'epoch': 6.0}


In [38]:
chinese_test_results = trainer.evaluate(chinese_test_tokd)
print(chinese_test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 2.9251914024353027, 'eval_accuracy': 0.5653333333333334, 'eval_precision': 0.5558016498436344, 'eval_recall': 0.5653333333333334, 'eval_f1': 0.5080874042027337, 'eval_runtime': 18.0142, 'eval_samples_per_second': 166.535, 'eval_steps_per_second': 5.218, 'epoch': 6.0}


In [39]:
chinese_valid_results = trainer.evaluate(chinese_valid_tokd)
print(chinese_valid_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 2.9991276264190674, 'eval_accuracy': 0.548, 'eval_precision': 0.5227310852001389, 'eval_recall': 0.548, 'eval_f1': 0.4888098522873773, 'eval_runtime': 17.9656, 'eval_samples_per_second': 166.986, 'eval_steps_per_second': 5.232, 'epoch': 6.0}


In [40]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
test_preds = trainer.predict(test_tokd)

y_true_test = test_preds.label_ids
y_pred_test = np.argmax(test_preds.predictions, axis=1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [ ]:
val_preds = trainer.predict(val_tokd)

y_true_val = val_preds.label_ids
y_pred_val = np.argmax(val_preds.predictions, axis=1)

In [ ]:
train_preds = trainer.predict(train_tokd)

y_true_train = train_preds.label_ids
y_pred_train = np.argmax(train_preds.predictions, axis=1)

In [ ]:
jap_preds = trainer.predict(jap_tokd)

y_true_jap = jap_preds.label_ids
y_pred_jap = np.argmax(jap_preds.predictions, axis=1)

In [ ]:
jap_test_preds = trainer.predict(jap_test_tokd)

y_true_jap_test = jap_test_preds.label_ids
y_pred_jap_test = np.argmax(jap_test_preds.predictions, axis=1)

In [ ]:
chinese_test_preds = trainer.predict(chinese_test_tokd)

y_true_chinese_test = chinese_test_preds.label_ids
y_pred_chinese_test = np.argmax(chinese_test_preds.predictions, axis=1)

In [ ]:
chinese_valid_preds = trainer.predict(chinese_valid_tokd)

y_true_chinese_valid = chinese_valid_preds.label_ids
y_pred_chinese_valid = np.argmax(chinese_valid_preds.predictions, axis=1)

In [ ]:
cm_train = confusion_matrix(y_true_train, y_pred_train)
print("Confusion Matrix (Train):\n", cm_train)

In [ ]:
cm_test = confusion_matrix(y_true_test, y_pred_test)
print("Confusion Matrix (Test):\n", cm_test)

In [ ]:
cm_chinese_test = confusion_matrix(y_true_chinese_test, y_pred_chinese_test)
print("Confusion Matrix (Chinese Test):\n", cm_chinese_test)

In [ ]:
cm_chinese_valid = confusion_matrix(y_true_chinese_valid, y_pred_chinese_valid)
print("Confusion Matrix (Chinese Valid):\n", cm_chinese_valid)

In [ ]:
cm_test_jap = confusion_matrix(y_true_jap_test, y_pred_jap_test)
print("Confusion Matrix (Japenese Test):\n", cm_test_jap)

In [ ]:
cm_val = confusion_matrix(y_true_val, y_pred_val)
print("Confusion Matrix (Valid):\n", cm_val)

In [ ]:
cm_jap = confusion_matrix(y_true_jap, y_pred_jap)
print("Confusion Matrix (Japenese Valid):\n", cm_jap)

In [ ]:
def get_tp_tn_fp_fn(cm):
    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    TN = cm.sum() - (TP + FP + FN)
    
    return TP, TN, FP, FN

In [ ]:
TP, TN, FP, FN = get_tp_tn_fp_fn(cm_test)

print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_test_jap, 
            annot=True,        # show numbers
            fmt='d',           # integer format
            cmap='Blues',      # color
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Japenese Test Data")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_test, 
            annot=True,        # show numbers
            fmt='d',           # integer format
            cmap='Blues',      # color
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Test Data")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_val, 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Validation Data")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_jap, 
            annot=True, 
            fmt='d', 
            cmap='Greens',   # different color for variety
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Japanese Valid Dataset")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_test_jap, 
            annot=True,        # show numbers
            fmt='d',           # integer format
            cmap='Blues',      # color
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Japenese Test Data")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_chinese_test, 
            annot=True,        # show numbers
            fmt='d',           # integer format
            cmap='Blues',      # color
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Chinese Test Data")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm_chinese_valid, 
            annot=True,        # show numbers
            fmt='d',           # integer format
            cmap='Blues',      # color
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Chinese Valid Data")

plt.show()